In [46]:
from paddleocr import TextLineOrientationClassification
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image
import numpy as np
from pathlib import Path

In [32]:
model025 = TextLineOrientationClassification(model_name="PP-LCNet_x0_25_textline_ori", engine="transformers")
model10 = TextLineOrientationClassification(model_name="PP-LCNet_x1_0_textline_ori", engine="transformers")

Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Tea\.paddlex\official_models\PP-LCNet_x0_25_textline_ori_safetensors`.


Loading weights: 100%|██████████| 146/146 [00:00<00:00, 5426.39it/s]
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Tea\.paddlex\official_models\PP-LCNet_x1_0_textline_ori_safetensors`.
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 4711.72it/s]


In [33]:
df = pd.read_csv("../sample_submission.csv")
df.head()

,image_id,p_180
0,test_00000,0.5
1,test_00001,0.5
2,test_00002,0.5
3,test_00003,0.5
4,test_00004,0.5


In [44]:
def p180(model, image):
    result = model.predict(image, batch_size=1)[0]
    label = np.asarray(result["label_names"]).reshape(-1)[0]
    score = float(np.asarray(result["scores"]).reshape(-1)[0])

    if label == "180_degree":
        return score
    if label == "0_degree":
        return 1 - score
    raise ValueError(f"Unexpected label: {label}")

results = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    image_id = row["image_id"]

    with Image.open(f"../test/images/{image_id}.png") as image:
        rgb = np.array(image.convert("RGB"))

    # RGB → BGR; preserve HWC layout.
    img = rgb[:, :, ::-1].copy()
    img_rotated = np.rot90(img, 2).copy()

    p025 = p180(model025, img)
    p025rot = p180(model025, img_rotated)

    p10 = p180(model10, img)
    p10rot = p180(model10, img_rotated)

    q = (p025 + (1 - p025rot) + p10 + (1 - p10rot)) / 4
    q = np.clip(q, 1e-6, 1 - 1e-6)

    pred = q**2 / (q**2 + (1 - q)**2)
    results.append(float(pred))

100%|██████████| 20000/20000 [05:59<00:00, 55.56it/s]


In [45]:
pd.DataFrame({"image_id": df["image_id"], "p_180": results}).to_csv("../labels_paddleensemble.csv", index=False)

In [47]:
benchmark = Path("E:/projects/Avito-BootCamp-FastTrack/artifacts/hiertext")
manifest = pd.read_csv(benchmark / "manifest.csv")

split = "development"

# Each saved crop is upright. Evaluate it and its 180° counterpart once.
crops = manifest[
    (manifest["split"] == split)
    & (manifest["cohort"] == "matched")
    & (manifest["rotation"] == 0)
]

results = []

for row in tqdm(crops.itertuples(index=False), total=len(crops)):    
    with Image.open(str(benchmark / row.image_path)) as image:
        rgb = np.array(image.convert("RGB"))

    # RGB → BGR; preserve HWC layout.
    img = rgb[:, :, ::-1].copy()
    img_rotated = np.rot90(img, 2).copy()

    a = p180(model025, img)
    ar = p180(model025, img_rotated)
    b = p180(model10, img)
    br = p180(model10, img_rotated)

    q = (a + (1 - ar) + b + (1 - br)) / 4
    q = np.clip(q, 1e-6, 1 - 1e-6)
    ensemble = q**2 / (q**2 + (1 - q)**2)

    results.extend([
        [row.pair_id, 0, a, b, ensemble],
        [row.pair_id, 1, ar, br, 1 - ensemble],
    ])

results = pd.DataFrame(
    results,
    columns=["pair_id", "label", "paddle025", "paddle10", "ensemble"],
)

for name in ["paddle025", "paddle10", "ensemble"]:
    p = results[name].to_numpy()
    y = results["label"].to_numpy()

    score = 1 - np.mean((p - y)**2)
    accuracy = np.mean((p >= 0.5) == y)

    print(f"{name:12s}  1-Brier={score:.6f}  accuracy={accuracy:.2%}")

100%|██████████| 2230/2230 [00:37<00:00, 59.78it/s]

paddle025     1-Brier=0.896817  accuracy=86.97%
paddle10      1-Brier=0.818348  accuracy=78.23%
ensemble      1-Brier=0.957946  accuracy=96.50%
